# Trimming the Lab Event Data (Version 2)
The objective of this notebook is similar to *trim_labevents.ipynb*. Namely, starting with *labevents.csv* from MIMIC-IV, two reductions are made. The first reduction entails eliminating all of the data associated with patients that **do not** have kidney disease. However, in contrast to *trim_labevents.ipynb*, this notebook focuses on the 11 diagnoses that are specified in *Nephropathy Project.pdf*. Importantly, none of these diagnoses were considered previously. The second reduction entails eliminating all of the data that **is not** associated with one of the labs in *Minimal Reducedlabitems - Reducedlabitems.csv*. Note that this list is a shortened version of the list that was used in *trim_labevents.ipynb*.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

The next cell implements the first step of the reduction. *labevents.csv* is processed in chunks, where only the data that corresponds to a patient with kidney disease is kept. The result from this step is the dataframe `reduced_labevents_df1`.

In [ ]:
diagnoses_df = pd.read_csv('../Data/diagnoses_icd_reduced.csv')
subject_ids = list(set(diagnoses_df['subject_id']))
labevents_cols = ['labevent_id',
                 'subject_id',
                 'hadm_id',
                 'specimen_id',
                 'itemid',
                 'order_provider_id',
                 'charttime',
                 'storetime',
                 'value',
                 'valuenum',
                 'valueuom',
                 'ref_range_lower',
                 'ref_range_upper',
                 'flag',
                 'priority',
                 'comments']
reduced_labevents_df1 = pd.DataFrame(columns = labevents_cols)
chunk_size = 10**6
for chunk in pd.read_csv('../Data/labevents.csv', chunksize = chunk_size):
    for patient in subject_ids:
        chunk_reduced = chunk[chunk['subject_id'] == patient]
        reduced_labevents_df1 = pd.concat([reduced_labevents_df1, chunk_reduced])


The final cell (below) implements the second step of the reduction. Starting with `reduced_labevents_df1`, all of the data pertaining to labs that are not pertinent to our study are eliminated. The result from this step is the dataframe `reduced_labevents_df2`. This dataframe is written to the csv file called *kidney_disease_patients_version2.csv*.

In [ ]:
reduced_labitems_df = pd.read_csv('../Data/Minimal Reducedlabitems - Reducedlabitems.csv')
item_ids = list(reduced_labitems_df['itemid'])
reduced_labevents_df2 = pd.DataFrame(columns = labevents_cols)
for lab in item_ids:
    lab_df = reduced_labevents_df1.loc[reduced_labevents_df1['itemid'] == lab]
    reduced_labevents_df2 = pd.concat([reduced_labevents_df2, lab_df])
reduced_labevents_df2.to_csv('../Data/kidney_disease_patients_version2.csv', index = False)